# CA4 Question 1: RNN Models for Joint Spoken Language Understanding (SLU)

## Intent Detection & Slot Filling with ATIS Dataset

This notebook implements:
- 1.1 Data Exploration & Preparation
- 1.2 BiRNN Baseline for Slot Filling
- 1.3 BiLSTM Joint Model (Intent + Slots)
- 1.4 Encoder-Decoder Non-aligned Joint Model

In [ ]:
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict, Tuple
from collections import defaultdict

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_

import seqeval
from seqeval.metrics import classification_report, f1_score

# Add src to path
sys.path.insert(0, str(Path('.').resolve() / 'src'))
from data.preprocess import (
    load_atis_examples, build_vocab, build_label_vocab, 
    ATISDataset, collate_fn, WhitespaceTokenizer, PAD, UNK
)
from models.baseline import BiRNNSlotFiller, BiLSTMJoint
from models.encoder_decoder import Encoder, Decoder, Seq2SeqJoint
from utils.metrics import slot_f1, slot_classification_report, intent_accuracy

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 1.1 Data Preparation: Load ATIS Dataset

Download from Kaggle and explore the dataset structure.
```bash
kagglehub dataset download "siddhadev/atis-dataset-clean"
```

If not available, we'll load sample data or create synthetic examples.

In [ ]:
# Try to download dataset from Kaggle (requires kagglehub installed)
try:
    import kagglehub
    atis_path = kagglehub.dataset_download("siddhadev/atis-dataset-clean")
    print(f"✓ ATIS dataset downloaded to: {atis_path}")
except Exception as e:
    print(f"⚠ Could not download from Kaggle: {e}")
    # Create a path for local dataset
    atis_path = Path("./data/atis")
    atis_path.mkdir(parents=True, exist_ok=True)
    print(f"Will use local path: {atis_path}")

# Load dataset from files
try:
    train_examples = load_atis_examples(atis_path / "train.json" if (atis_path / "train.json").exists() 
                                        else atis_path)
    test_examples = load_atis_examples(atis_path / "test.json" if (atis_path / "test.json").exists() 
                                       else atis_path)
    
    # If only single file, split into train/val/test
    if len(train_examples) == 0 and len(test_examples) == 0:
        all_ex = load_atis_examples(atis_path)
        n = len(all_ex)
        random.shuffle(all_ex)
        train_examples = all_ex[:int(0.7*n)]
        val_examples = all_ex[int(0.7*n):int(0.9*n)]
        test_examples = all_ex[int(0.9*n):]
    else:
        # Try to load validation set
        val_examples = load_atis_examples(atis_path / "dev.json" if (atis_path / "dev.json").exists() 
                                          else atis_path) if len(test_examples) > 0 else []
        if not val_examples:
            # Split test into val/test
            n_test = len(test_examples)
            val_examples = test_examples[:n_test//2]
            test_examples = test_examples[n_test//2:]
            
except Exception as e:
    print(f"⚠ Could not load dataset: {e}")
    # Create minimal synthetic dataset for testing
    train_examples = [
        {"tokens": ["show", "me", "flights", "from", "boston", "to", "denver"],
         "slots": ["O", "O", "O", "O", "B-fromloc", "O", "B-toloc"],
         "intent": "atis_flight"},
        {"tokens": ["what", "is", "the", "fare", "for", "this", "flight"],
         "slots": ["O", "O", "O", "B-fare", "O", "O", "O"],
         "intent": "atis_airfare"},
    ]
    val_examples = [d.copy() for d in train_examples]
    test_examples = [d.copy() for d in train_examples]

print(f"\n📊 Dataset Statistics:")
print(f"  Train examples: {len(train_examples)}")
print(f"  Val examples: {len(val_examples)}")
print(f"  Test examples: {len(test_examples)}")

# Count intents and slots
intents = set()
slots = set()
for ex in train_examples + val_examples + test_examples:
    intents.add(ex["intent"])
    slots.update(ex["slots"])

print(f"  Unique intents: {len(intents)}")
print(f"  Unique slot labels: {len(slots)}")
print(f"\n  Intents: {sorted(intents)[:5]}... ({len(intents)} total)")
print(f"  Slots: {sorted(slots)}")

# Show sample
print(f"\n📝 Sample example:")
print(f"  Tokens: {train_examples[0]['tokens']}")
print(f"  Slots:  {train_examples[0]['slots']}")
print(f"  Intent: {train_examples[0]['intent']}")